## Chargement et reprise


In [4]:
import json, re, urllib.request
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


with open("../data/chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)
with open("../data/questions_test.json", encoding="utf-8") as f:
    questions_test = json.load(f)

ids = [c["chunk_id"] for c in chunks]

def tokeniser(texte):
    mots = re.findall(r"[a-z0-9]+", texte.lower())
    return [m for m in mots if m not in ENGLISH_STOP_WORDS]

bm25 = BM25Okapi([tokeniser(c["texte"]) for c in chunks])

def rechercher_bm25(question, k=5):
    scores = bm25.get_scores(tokeniser(question))
    ordre = np.argsort(-scores)[:k]
    return [(ids[i], float(scores[i])) for i in ordre]
modele = SentenceTransformer("all-MiniLM-L6-v2")
E = np.load("../data/embeddings_minilm.npy")

def rechercher_dense(question, k=5):
    q = modele.encode([question], normalize_embeddings=True)[0]
    scores = E @ q
    ordre = np.argsort(-scores)[:k]
    return [(ids[i], float(scores[i])) for i in ordre]

def rappel_a_k(fonction, k):
    qs = [q for q in questions_test if q["chunks_attendus"]]
    ok = [any(cid in [i for i, _ in fonction(q["q"], k)] for cid in q["chunks_attendus"]) for q in qs]
    return sum(ok) / len(qs)

def minmax(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-12)

def scores_hybrides(question, alpha):
    s_bm25 = bm25.get_scores(tokeniser(question))
    s_dense = E @ modele.encode([question], normalize_embeddings=True)[0]
    return alpha * minmax(s_dense) + (1 - alpha) * minmax(s_bm25)

def rechercher_hybride(question, k=5, alpha=0.5):
    s = scores_hybrides(question, alpha)
    ordre = np.argsort(-s)[:k]
    return [(ids[i], float(s[i])) for i in ordre]

def meilleur_score_dense(question):
    return rechercher_dense(question, 1)[0][1]

par_id = {c["chunk_id"]: c for c in chunks}


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Rappel@k


In [ ]:
hyb = lambda q, k: rechercher_hybride(q, k, 0.8)
print("Méthode   |  k=1   k=3   k=5   k=10")
for nom, f in [("BM25", rechercher_bm25), ("Dense", rechercher_dense), ("Hybride", hyb)]:
    print(f"{nom:9} |", "  ".join(f"{rappel_a_k(f, k):.2f}" for k in (1, 3, 5, 10)))


## Choix de l’alpha


In [ ]:
print("alpha |  k=1   k=3   k=5   k=10")
for alpha in [0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]:
    f = lambda q, k, a=alpha: rechercher_hybride(q, k, a)
    print(f"{alpha:5} |", "  ".join(f"{rappel_a_k(f, k):.2f}" for k in (1, 3, 5, 10)))
ALPHA = 0.8
print("Alpha retenu :", ALPHA)


## Choix du seuil


In [ ]:
avec = [meilleur_score_dense(q["q"]) for q in questions_test if q["chunks_attendus"]]
sans = [meilleur_score_dense(q["q"]) for q in questions_test if not q["chunks_attendus"]]

sc_avec = np.array(avec)
sc_sans = np.array(sans)

print("seuil | refus justifiés (sur 5) | refus injustifiés (sur 25)")
for s in [0.45, 0.50, 0.53, 0.55, 0.57, 0.60, 0.65]:
    just = int((sc_sans < s).sum())
    injust = int((sc_avec < s).sum())
    print(f"{s:5} |          {just}/5            |          {injust}/25")

SEUIL = 0.57
print("Seuil retenu :", SEUIL)


## Règles et avertissements


In [ ]:
REFUS = "I don't know based on the available documents."
MOTIF_DOSE_Q = re.compile(r"\bdose\b|dosage|how many (?:milliliters|ml|mg|pills|tablets)|how much|how often|how many days|amounts?\b", re.I)
MOTIF_DOSE_T = re.compile(r"\d+(?:[.,]\d+)?\s?(?:mg|mcg|g|ml)(?!/dl)\b", re.I)
MOTIF_GROSSESSE = re.compile(r"pregnan|expecting a baby|trimester|breastfe", re.I)
MOTIF_ENFANT = re.compile(r"\bchild|\bchildren|infant|baby|toddler|newborn|teenager|\bkids?\b|year-old", re.I)

def avertissements(question):
    a = []
    if MOTIF_GROSSESSE.search(question):
        a.append("Pregnancy: consult a healthcare professional.")
    if MOTIF_ENFANT.search(question):
        a.append("Child: consult a healthcare professional.")
    return a


## Génération avec Ollama


In [ ]:
MODELE = "llama3.2:3b"

def appeler_ollama(consigne):
    req = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=json.dumps({"model": MODELE, "prompt": consigne, "stream": False,
                         "options": {"temperature": 0}}).encode(),
        headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.loads(r.read())["response"].strip()

def construire_consigne(question, passages):
    contexte = "\n\n".join(f"[{n+1}] {par_id[pid]['texte']}" for n, pid in enumerate(passages))
    return ("You are a medical information assistant. Use the passages below to answer the question. "
            "If the passages contain information that answers the question, even partly, "
            "answer using only that information and cite the passages as [1], [2] or [3]. "
            "Do not add anything that is not in the passages. "
            "Never give a dose unless it appears in a passage. "
            f"Only if none of the passages is related to the question, reply exactly: {REFUS}\n\n"
            f"Passages:\n{contexte}\n\nQuestion: {question}\nAnswer:")

def repondre_llm(question, k=3):
    passages = [i for i, _ in rechercher_hybride(question, k, ALPHA)]
    out = {"avertissements": avertissements(question), "citations": [], "passages": passages}

    if meilleur_score_dense(question) < SEUIL:
        return {**out, "reponse": REFUS, "raison": "seuil"}
    if MOTIF_DOSE_Q.search(question) and not any(MOTIF_DOSE_T.search(par_id[p]["texte"]) for p in passages):
        return {**out, "reponse": REFUS, "raison": "regle_dose"}

    rep = appeler_ollama(construire_consigne(question, passages))
    if rep.lower().startswith("i don't know"):
        return {**out, "reponse": REFUS, "raison": "refus_llm"}

    nums = sorted({int(n) for n in re.findall(r"\[(\d)\]", rep) if 1 <= int(n) <= len(passages)})
    cites = [passages[n - 1] for n in nums]
    if not cites:
        return {**out, "reponse": REFUS, "raison": "pas_de_citation", "brut": rep}
    if MOTIF_DOSE_T.search(rep) and not any(MOTIF_DOSE_T.search(par_id[c]["texte"]) for c in cites):
        return {**out, "reponse": REFUS, "raison": "dose_non_sourcee", "brut": rep}
    return {**out, "reponse": rep, "raison": "ok", "citations": cites}


## 30 questions


In [ ]:
resultats_v2 = []
for n, q in enumerate(questions_test):
    r = repondre_llm(q["q"])
    resultats_v2.append({"q": q["q"], "attendu": q["chunks_attendus"], **r})
    print(n + 1, "/ 30 |", r["raison"])

with open("../resultats/resultats_rag_v2.json", "w", encoding="utf-8") as f:
    json.dump(resultats_v2, f, ensure_ascii=False, indent=1)

avec_rep = [r for r in resultats_v2 if r["attendu"]]
sans_rep = [r for r in resultats_v2 if not r["attendu"]]
print("\nRefus justifiés   :", sum(r["raison"] != "ok" for r in sans_rep), "/ 5")
print("Refus injustifiés :", sum(r["raison"] != "ok" for r in avec_rep), "/ 25")
rep = [r for r in avec_rep if r["raison"] == "ok"]
print("Citation correcte :", sum(any(c in r["attendu"] for c in r["citations"]) for r in rep), "/", len(rep))


## Référence sans RAG


In [ ]:
def repondre_sans_rag(question):
    consigne = ("You are a medical information assistant. Answer the question briefly.\n\n"
                f"Question: {question}\nAnswer:")
    return appeler_ollama(consigne)

resultats_ref = []
for n, q in enumerate(questions_test):
    rep = repondre_sans_rag(q["q"])
    refus = bool(re.search(r"i don't know|i do not know|cannot|can't|not able to", rep, re.I))
    donne_dose = bool(MOTIF_DOSE_T.search(rep))
    resultats_ref.append({"q": q["q"], "attendu": q["chunks_attendus"], "reponse": rep,
                          "refus": refus, "dose": donne_dose})
    print(n + 1, "/ 30 | refus:", refus, "| dose:", donne_dose)

with open("../resultats/resultats_sans_rag.json", "w", encoding="utf-8") as f:
    json.dump(resultats_ref, f, ensure_ascii=False, indent=1)

sans = [r for r in resultats_ref if not r["attendu"]]
print("\nSans RAG, questions sans réponse refusées :", sum(r["refus"] for r in sans), "/ 5")
print("Sans RAG, réponses avec une dose chiffrée (sans réponse) :", sum(r["dose"] for r in sans), "/ 5")
print("Sans RAG, questions avec réponse refusées :", sum(r["refus"] for r in resultats_ref if r["attendu"]), "/ 25")


## Fidélité


In [ ]:
with open("../resultats/resultats_rag_v2.json", encoding="utf-8") as f:
    resultats_v2 = json.load(f)

a_noter = [r for r in resultats_v2 if r["raison"] == "ok"]
print(len(a_noter), "réponses à noter\n")

def afficher(i):
    r = a_noter[i]
    print(f"=== Réponse {i+1}/{len(a_noter)} ===")
    print("Q :", r["q"])
    print("\nRÉPONSE :", r["reponse"])
    for n, cid in enumerate(r["passages"]):
        print(f"\n[{n+1}] {cid} :", par_id[cid]["texte"][:700])
    print("\n" + "=" * 60)

afficher(0)


In [ ]:
notes = [0.5, 0.5, 0.5, 1, 1, 0.5, 0.5, 1, 1, 0.5, 1, 0.5, 1, 1, 0.5, 1, 0.5, 1, 1]
print(len(notes), "notes | fidélité moyenne :", round(sum(notes)/len(notes), 2))

with open("../resultats/notes_fidelite.json", "w") as f:
    json.dump(notes, f)
